# Tables

All tables for the paper. Each is displayed here and written to `../tables/` as `.csv` and a booktabs `.tex` fragment.

Most results are read from the CSVs written by `02_training_and_testing_the_ann.ipynb` and `03_ann_vs_linear_models.ipynb` (in `tables/`, next to the tables built from them). The free-slope circular (Möbius) baselines, and the leave-one-wind-speed-out baselines, are not among those results, so they are fit here with the same trajectory-grouped split (seed 42, 33% test) and the same evaluation protocol as the OLS baselines.

1. Linear / circular baselines and the ANN, by regime
2. Per-trajectory error distribution, all models
3. ANN error by regime
4. Leave-one-wind-speed-out (ANN and baselines)
5. Robustness across training seeds
6. Architecture selection (grouped CV)
7. Ensemble-agreement calibration
8. ANN error vs. trajectory covariates

In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
sys.path.insert(0, os.path.abspath('../utils'))
from utils import circular_distance
from IPython.display import display

D = '../trainingdata/04_trainingdata/'
OUT = '../tables/'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
REGIMES = ['upwind', 'crosswind', 'downwind']
THRESH = 15.0

def classify_direction(a):
    deg = np.degrees(np.arctan2(np.sin(a), np.cos(a)))
    return np.where(np.abs(deg) <= 45, 'downwind', np.where(np.abs(np.abs(deg) - 180) <= 45, 'upwind', 'crosswind'))

def bootstrap_ci(v, n_boot=5000, seed=0):
    v = np.asarray(v); rng = np.random.default_rng(seed)
    med = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    lo, hi = np.percentile(med, [2.5, 97.5])
    return float(np.median(v)), float(lo), float(hi)

def ci_str(m, lo, hi): return f'{m:.2f} [{lo:.2f}, {hi:.2f}]'

def esc(s): return str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

def save(df, name, caption, label, align=None):
    df.to_csv(f'{OUT}{name}.csv', index=False)
    align = align or 'l' + 'r' * (df.shape[1] - 1)
    lines = [r'\begin{table}[h!]', r'    \centering', f'    \\caption{{{caption}}}', f'    \\label{{tab:{label}}}',
             f'    \\begin{{tabular}}{{@{{}}{align}@{{}}}}', r'        \toprule',
             '        ' + ' & '.join(esc(c) for c in df.columns) + r' \\', r'        \midrule']
    for _, r in df.iterrows():
        lines.append('        ' + ' & '.join(esc(x) for x in r.values) + r' \\')
    lines += [r'        \bottomrule', r'    \end{tabular}', r'\end{table}']
    open(f'{OUT}{name}.tex', 'w').write('\n'.join(lines) + '\n')
    display(df)

## Setup: split, and linear / circular / Möbius baselines

Baselines: *linear* OLS regresses the heading unit vector on the Cartesian components of each kinematic vector; *circular* OLS regresses it on the unit direction only (degree-1 trigonometric polynomial). The additional **Möbius** model is the free-slope circular-circular regression from `03_ann_vs_linear_models.ipynb`: $\hat\theta = \mu + 2\arctan(\beta\tan\frac{\theta_x-\nu}{2})$, fit by minimising $\sum(1-\cos e)$. It takes a single input angle, so it is fit once per kinematic vector.

In [2]:
data = pd.read_csv(D + 'all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv')
ids = data['trajec_objid'].unique()
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(np.zeros((len(ids), 1)), groups=ids))
train_df = data[data['trajec_objid'].isin(set(ids[tr]))].reset_index(drop=True)
test_df  = data[data['trajec_objid'].isin(set(ids[te]))].reset_index(drop=True)
print(len(train_df), 'train frames,', len(test_df), 'test frames')

QUANT = {'thrust': ('thrust', 'thrust_angle'), 'air': ('airspeed', 'airspeed_angle'), 'ground': ('groundspeed', 'groundspeed_angle')}

def design(d, combo, kind):
    cols = [np.ones(len(d))]
    for q in combo:
        m, a = QUANT[q]
        if kind == 'linear': cols += [d[m].values * np.cos(d[a].values), d[m].values * np.sin(d[a].values)]
        else:                cols += [np.cos(d[a].values), np.sin(d[a].values)]
    return np.column_stack(cols)

def ols_angle(train, test, combo, kind):
    Xtr, Xte = design(train, combo, kind), design(test, combo, kind)
    coef = np.linalg.lstsq(Xtr, train[['heading_angle_x', 'heading_angle_y']].values, rcond=None)[0]
    p = Xte @ coef
    return np.arctan2(p[:, 1], p[:, 0])

def circ_mean(a): return np.arctan2(np.sin(a).mean(), np.cos(a).mean())
def mobius(x, mu, nu, lb): return mu + 2 * np.arctan(np.exp(lb) * np.tan((x - nu) / 2))

def fit_mobius(x, y):
    best = None
    for nu0 in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        for lb0 in (-1.0, 0.0, 1.0, 2.0):
            r = minimize(lambda p: np.mean(1 - np.cos(y - mobius(x, *p))), [circ_mean(y - x), nu0, lb0],
                         method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-10, maxiter=4000))
            if best is None or r.fun < best.fun: best = r
    return best.x

def mobius_angle(train, test, q):
    a = QUANT[q][1]
    return mobius(test[a].values, *fit_mobius(train[a].values, train['heading_angle'].values))

def frame_errors(test, pred_angle):
    err = np.degrees(circular_distance(test['heading_angle'].values, pred_angle))
    return pd.DataFrame({'trajec_objid': test['trajec_objid'].values, 'err': err,
                         'regime': classify_direction(test['heading_angle'].values)})

def summarize(fdf):
    """per-trajectory medians (pooled and per regime), as in the paper"""
    pooled = fdf.groupby('trajec_objid')['err'].median()
    by_reg = fdf.groupby(['regime', 'trajec_objid'])['err'].median().reset_index()
    out = {'overall': pooled.median()}
    for r in REGIMES: out[r] = by_reg.loc[by_reg['regime'] == r, 'err'].median()
    return out, pooled, by_reg

# Möbius baselines on the paper's split
mobius_rows, mobius_traj = [], {}
for q in QUANT:
    s, pooled, _ = summarize(frame_errors(test_df, mobius_angle(train_df, test_df, q)))
    name = f'mobius: {q}'
    mobius_rows.append({'model': name, **s}); mobius_traj[name] = pooled
print(pd.DataFrame(mobius_rows))

# sanity check: reproduce one saved OLS baseline
ref = pd.read_csv(OUT + 'ols_baselines_by_regime.csv').set_index('model').loc['linear: thrust+air', 'overall']
chk, _, _ = summarize(frame_errors(test_df, ols_angle(train_df, test_df, ('thrust', 'air'), 'linear')))
print(f"check linear: thrust+air overall: saved {ref:.3f}, recomputed {chk['overall']:.3f}")

76331 train frames, 37555 test frames


            model    overall     upwind  crosswind   downwind
0  mobius: thrust  20.013671  13.369395  49.647565  69.088360
1     mobius: air   7.931983   6.677876  25.776554  30.202400
2  mobius: ground  17.486064   8.830068  44.873058  43.862108
check linear: thrust+air overall: saved 9.473, recomputed 9.473


## Table 1 — Linear, circular and Möbius baselines vs. the ANN, by flight regime

Median per-trajectory angular error (deg) on the held-out test set. Regimes are classified from the true heading relative to the wind axis (downwind $|\theta|\le45^\circ$, upwind within $45^\circ$ of $180^\circ$, otherwise crosswind; the wind blows toward +x).

In [3]:
ann_frames = pd.read_csv(OUT + 'ann_by_regime.csv')
ann_s, ann_pooled, ann_by_reg = summarize(ann_frames)
base = pd.read_csv(OUT + 'ols_baselines_by_regime.csv')
t1 = pd.concat([base, pd.DataFrame(mobius_rows), pd.DataFrame([{'model': 'ANN', **ann_s}])], ignore_index=True)
t1['family'] = t1['model'].str.split(':').str[0].map({'linear': 'Linear OLS', 'circular': 'Circular OLS',
                                                     'mobius': 'Möbius circular regression', 'ANN': 'ANN'})
t1['inputs'] = t1['model'].str.split(': ').str[-1].where(t1['model'] != 'ANN', 'groundspeed + airspeed')
t1 = t1.sort_values('overall', ascending=False).reset_index(drop=True)
table1 = t1[['family', 'inputs', 'overall', 'upwind', 'crosswind', 'downwind']].copy()
table1.columns = ['Model', 'Inputs', 'Overall', 'Upwind', 'Crosswind', 'Downwind']
best_idx = {c: t1[t1['family'] != 'ANN'][c.lower()].idxmin() for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']}
print('best non-ANN baseline per column:', {c: (t1.loc[i, 'model'], round(t1.loc[i, c.lower()], 2)) for c, i in best_idx.items()})
for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']: table1[c] = table1[c].map('{:.2f}'.format)
save(table1, 'table1_baselines_by_regime',
     'Median per-trajectory angular error (deg) of every linear, circular and M\\"obius baseline and of the ANN on the held-out test trajectories, overall and by flight regime. Sorted by overall error, worst first.',
     'baseline_regime', align='llrrrr')

best non-ANN baseline per column: {'Overall': ('mobius: air', np.float64(7.93)), 'Upwind': ('linear: thrust+air+ground', np.float64(5.36)), 'Crosswind': ('mobius: air', np.float64(25.78)), 'Downwind': ('mobius: air', np.float64(30.2))}


,Model,Inputs,Overall,Upwind,Crosswind,Downwind
0,Möbius circular regression,thrust,20.01,13.37,49.65,69.09
1,Linear OLS,thrust,18.73,11.35,57.16,115.16
2,Circular OLS,ground,18.65,8.05,37.93,91.54
3,Circular OLS,thrust,18.35,11.59,55.06,77.17
4,Möbius circular regression,ground,17.49,8.83,44.87,43.86
5,Circular OLS,thrust+ground,12.50,7.77,36.90,50.22
6,Linear OLS,ground,11.36,5.63,34.78,50.30
7,Linear OLS,air+ground,11.02,5.50,34.38,46.06
8,Circular OLS,air,10.65,6.29,43.68,41.02
9,Circular OLS,thrust+air,10.62,6.78,41.11,41.94


## Table 2 — Per-trajectory error distribution, all models

Median with 95% bootstrap CI (resampling trajectories), mean, interquartile range, 90th percentile and fraction of trajectories with error above 15°.

In [4]:
def dist_row(name, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Model': name, 'n': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', 'Q90': f'{np.percentile(v, 90):.2f}',
            f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

per_traj_all = pd.read_csv(OUT + 'trajectory_level_errors_all_models.csv')
rows = [dist_row(m, g['err']) for m, g in per_traj_all.groupby('model')]
rows += [dist_row(m, v.values) for m, v in mobius_traj.items()]
table2 = pd.DataFrame(rows)
table2['_k'] = table2['Median [95% CI]'].str.split(' ').str[0].astype(float)
table2 = table2.sort_values('_k', ascending=False).drop(columns='_k').reset_index(drop=True)
save(table2, 'table2_error_distribution',
     'Distribution of per-trajectory median angular error (deg) on the held-out test set for the ANN and all baselines.',
     'error_distribution', align='lrrrrrr')

,Model,n,Median [95% CI],Mean,IQR,Q90,% > 15°
0,mobius: thrust,1102,"20.01 [18.62, 21.85]",28.66,9.83–41.21,65.42,59.8
1,linear: thrust,1102,"18.73 [16.67, 20.21]",31.03,8.08–43.27,77.33,56.5
2,circular: ground,1102,"18.65 [15.33, 22.14]",34.53,4.81–62.18,90.87,53.4
3,circular: thrust,1102,"18.35 [16.56, 19.97]",27.94,7.98–42.13,66.40,55.7
4,mobius: ground,1102,"17.49 [14.75, 19.38]",33.17,5.36–43.33,97.74,52.7
5,circular: thrust+ground,1102,"12.50 [11.49, 14.05]",22.76,4.92–30.84,58.73,45.4
6,linear: ground,1102,"11.36 [9.68, 13.22]",25.20,4.09–31.81,64.18,44.1
7,linear: air+ground,1102,"11.02 [9.45, 12.41]",24.02,4.06–29.62,64.24,43.2
8,circular: air,1102,"10.65 [9.22, 11.86]",22.84,3.93–29.59,61.09,42.0
9,circular: thrust+air,1102,"10.62 [9.71, 11.80]",22.17,4.40–28.46,58.32,40.6


## Table 3 — ANN error by flight regime

Per-trajectory median error within each regime (a trajectory contributes to every regime it visits), and the best baseline in that regime for reference.

In [5]:
def regime_row(label, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Regime': label, 'n trajectories': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

rows = [regime_row('Overall', ann_pooled.values)]
rows += [regime_row(r.capitalize(), ann_by_reg.loc[ann_by_reg['regime'] == r, 'err']) for r in REGIMES]
table3 = pd.DataFrame(rows)
nb_base = t1[t1['family'] != 'ANN']
table3['Best baseline (median)'] = [f"{nb_base[c].min():.2f}  ({nb_base.loc[nb_base[c].idxmin(), 'model']})"
                                    for c in ['overall'] + REGIMES]
save(table3, 'table3_ann_by_regime',
     'ANN per-trajectory angular error (deg) on the held-out test set, overall and by flight regime, with the best baseline median in each regime for comparison.',
     'ann_regime', align='lrrrrrl')

,Regime,n trajectories,Median [95% CI],Mean,IQR,% > 15°,Best baseline (median)
0,Overall,1099,"5.43 [5.06, 5.96]",19.26,2.91–14.24,23.7,7.93 (mobius: air)
1,Upwind,906,"5.29 [5.00, 5.83]",18.37,2.89–13.04,22.6,5.36 (linear: thrust+air+ground)
2,Crosswind,643,"21.99 [18.38, 27.71]",42.56,7.50–66.63,57.9,25.78 (mobius: air)
3,Downwind,258,"7.52 [4.42, 12.67]",53.87,2.57–125.63,41.9,30.20 (mobius: air)


## Table 4 — Leave-one-wind-speed-out

ANN: trained on two wind speeds and tested on the third (read from the saved results). The three baselines are refit here under the same leave-one-out design: the best pooled linear and circular OLS baselines and the airspeed Möbius regression. Baseline errors are medians over all frames of the held-out trajectories; ANN errors use the trajectories after the minimum-length filter, so counts can differ slightly.

In [6]:
lowo = pd.read_csv(OUT + 'leave_one_wind_out_results.csv')
BASELINES = [('Linear (thrust+air)', ('thrust', 'air'), 'linear'),
             ('Circular (thrust+air+ground)', ('thrust', 'air', 'ground'), 'circular'),
             ('Möbius (air)', ('air',), 'mobius')]
rows = []
for ws in sorted(data['windspeed'].unique()):
    trn, tst = data[data['windspeed'] != ws], data[data['windspeed'] == ws].reset_index(drop=True)
    res = {}
    for label, combo, kind in BASELINES:
        ang = mobius_angle(trn, tst, combo[0]) if kind == 'mobius' else ols_angle(trn, tst, combo, kind)
        s, _, _ = summarize(frame_errors(tst, ang)); res[label] = s
    for reg in ['overall'] + REGIMES:
        a = lowo[(np.isclose(lowo['held_out_windspeed'], ws)) & (lowo['regime'] == reg)].iloc[0]
        rows.append({'Held-out wind speed (m/s)': ws, 'Regime': reg.capitalize(), 'n (ANN)': int(a['n_trajectories']),
                     'ANN [95% CI]': ci_str(a['median'], a['ci95_lo'], a['ci95_hi']),
                     **{label: f'{res[label][reg]:.2f}' for label, _, _ in BASELINES}})
table4 = pd.DataFrame(rows)
save(table4, 'table4_leave_one_wind_out',
     'Leave-one-wind-speed-out median per-trajectory angular error (deg). The ANN and the three baselines are trained on two wind speeds and tested on the held-out third.',
     'lowo', align='lllrrrr')

,Held-out wind speed (m/s),Regime,n (ANN),ANN [95% CI],Linear (thrust+air),Circular (thrust+air+ground),Möbius (air)
0,0.3,Overall,1267,"5.95 [5.44, 6.47]",18.13,13.65,9.25
1,0.3,Upwind,924,"5.48 [4.93, 5.94]",6.43,7.63,7.67
2,0.3,Crosswind,784,"17.19 [15.14, 20.53]",36.41,36.55,19.43
3,0.3,Downwind,421,"8.54 [6.42, 12.44]",68.55,22.21,21.34
4,0.4,Overall,1063,"5.24 [4.77, 5.71]",10.12,10.37,8.71
5,0.4,Upwind,864,"5.03 [4.68, 5.58]",5.33,6.02,6.20
6,0.4,Crosswind,604,"17.18 [14.26, 20.74]",33.57,33.91,25.53
7,0.4,Downwind,252,"7.38 [5.13, 13.08]",56.29,30.29,29.89
8,0.6,Overall,999,"6.38 [5.83, 7.07]",7.00,9.34,6.91
9,0.6,Upwind,920,"5.41 [4.85, 5.78]",5.18,6.71,5.22


## Table 5 — Robustness across training seeds

In [7]:
seeds = pd.read_csv(OUT + 'ann_multiseed_results.csv')
t5 = pd.DataFrame({'Seed': seeds['seed'], 'Median error (deg)': seeds['median_error'].map('{:.2f}'.format),
                   'Mean error (deg)': seeds['mean_error'].map('{:.2f}'.format)})
summary = pd.DataFrame([{'Seed': 'mean ± sd', 'Median error (deg)': f"{seeds['median_error'].mean():.2f} ± {seeds['median_error'].std():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].mean():.2f} ± {seeds['mean_error'].std():.2f}"},
                        {'Seed': 'range', 'Median error (deg)': f"{seeds['median_error'].min():.2f}–{seeds['median_error'].max():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].min():.2f}–{seeds['mean_error'].max():.2f}"}])
table5 = pd.concat([t5, summary], ignore_index=True)
save(table5, 'table5_multiseed', 'ANN performance across five independently seeded training runs on the held-out test set.',
     'multiseed', align='lrr')

,Seed,Median error (deg),Mean error (deg)
0,42,5.43,19.26
1,7,5.62,18.82
2,13,5.40,18.29
3,99,5.35,18.43
4,123,5.42,18.93
5,mean ± sd,5.44 ± 0.10,18.75 ± 0.39
6,range,5.35–5.62,18.29–19.26


## Table 6 — Architecture selection (3-fold trajectory-grouped CV)

Out-of-fold median per-trajectory error with 95% bootstrap CI and IQR for each candidate architecture, trained with the geodesic loss and trajectory-length sample weights.

In [8]:
def cv_rows(df, label_fn):
    return [{'Configuration': label_fn(r), 'n': int(r['n_trajectories']),
             'CV median [95% CI]': ci_str(r['cv_median_error'], r['cv_ci95_lo'], r['cv_ci95_hi']),
             'IQR': f"{r['cv_q25']:.2f}–{r['cv_q75']:.2f}"} for _, r in df.iterrows()]

arch = lambda r: f"{int(r['layers'])} × {int(r['neurons'])}"
if os.path.exists(OUT + 'architecture_cv_results.csv'):
    table6 = pd.DataFrame(cv_rows(pd.read_csv(OUT + 'architecture_cv_results.csv'), arch))
    save(table6, 'table6_architecture_selection',
         'Grouped cross-validation results used to select the ANN architecture (geodesic loss, trajectory-length sample weights).',
         'selection', align='lrrr')
else:
    print('Skipping Table 6: architecture_cv_results.csv not found (run 02_training_and_testing_the_ann.ipynb with RETRAIN = True).')

Skipping Table 6: architecture_cv_results.csv not found (run 02_training_and_testing_the_ann.ipynb with RETRAIN = True).


## Table 7 — Ensemble-agreement calibration

In [9]:
ens = pd.read_csv(OUT + 'ann_ensemble_agreement.csv')
ens['quartile'] = pd.qcut(ens['R'], 4, labels=['Q1 (lowest agreement)', 'Q2', 'Q3', 'Q4 (highest agreement)'])
table7 = ens.groupby('quartile', observed=True).agg(n=('err', 'size'), median_R=('R', 'median'),
                                                     median_err=('err', 'median'), mean_err=('err', 'mean')).reset_index()
rho = spearmanr(ens['R'], ens['err']).statistic
table7.columns = ['Agreement quartile', 'n', 'Median R', 'Median error (deg)', 'Mean error (deg)']
table7['Median R'] = table7['Median R'].map('{:.5f}'.format)
for c in ['Median error (deg)', 'Mean error (deg)']: table7[c] = table7[c].map('{:.2f}'.format)
print(f'Spearman rho (R vs error) = {rho:.3f}')
save(table7, 'table7_ensemble_calibration',
     f'Per-trajectory error by quartile of five-model ensemble mean resultant length $R$ (Spearman $\\rho={rho:.2f}$).',
     'calibration', align='lrrrr')

Spearman rho (R vs error) = -0.635


,Agreement quartile,n,Median R,Median error (deg),Mean error (deg)
0,Q1 (lowest agreement),275,0.99830,19.60,45.43
1,Q2,276,0.99960,7.50,18.37
2,Q3,273,0.99978,4.46,8.37
3,Q4 (highest agreement),275,0.99988,2.66,4.79


## Table 8 — ANN error vs. trajectory covariates

Spearman rank correlation between per-trajectory ANN error and trajectory-level kinematic summaries on the held-out set, and median error by wind speed.

In [10]:
cov = pd.read_csv(OUT + 'ann_error_with_covariates.csv')
labels = {'mean_slip_angle': 'Mean |slip angle| (deg)', 'max_turning_rate': 'Max turning rate (deg/frame)',
          'mean_thrust': 'Mean thrust magnitude', 'mean_groundspeed': 'Mean groundspeed (m/s)',
          'mean_airspeed': 'Mean airspeed (m/s)', 'windspeed': 'Wind speed (m/s)'}
rows = []
for c, lab in labels.items():
    r = spearmanr(cov[c], cov['err'])
    rows.append({'Covariate': lab, 'Spearman ρ': f'{r.statistic:.3f}', 'p': f'{r.pvalue:.1e}', 'n': len(cov)})
table8 = pd.DataFrame(rows).sort_values('Spearman ρ', key=lambda s: s.astype(float).abs(), ascending=False).reset_index(drop=True)
save(table8, 'table8_covariate_correlations',
     'Spearman correlation between per-trajectory ANN error and trajectory covariates on the held-out test set.',
     'covariates', align='lrrr')
ws_tab = cov.groupby('windspeed')['err'].agg(n='size', median='median', mean='mean').round(2).reset_index()
display(ws_tab)

,Covariate,Spearman ρ,p,n
0,Mean |slip angle| (deg),0.721,9.0e-177,1099
1,Max turning rate (deg/frame),0.546,1.4e-86,1099
2,Mean groundspeed (m/s),-0.497,1.7e-69,1099
3,Mean airspeed (m/s),-0.486,3.3e-66,1099
4,Mean thrust magnitude,-0.395,2.6e-42,1099
5,Wind speed (m/s),-0.040,1.9e-01,1099


,windspeed,n,median,mean
0,0.3,400,5.23,19.76
1,0.4,370,5.64,18.76
2,0.6,329,5.56,19.19
